# FIRMS fire project: commented notebook

Pipeline, in order:
1. Download NASA FIRMS files for Brazil (MODIS 2000-2025, VIIRS NOAA-20 2018-2025).
2. Filter detections and aggregate them by 5 km cell and 7-day week.
3. Harmonize MODIS and VIIRS into one series (conversion factor by region and month).
4. Build the training table (features and labels for the next week).
5. Train and compare models (presence = classifier, intensity = regressor).
6. Export the model files for the API and check them.
7. Side checks: fire-days for Roraima, filtered detections export.

Only the comments and these text cells were added. The code is unchanged.
Cell outputs were removed to keep the file small; key results are written in the text cells.

## 1. Download the data

**Cell: install packages.** `remotezip` and `pyarrow` (Parquet files). They were already installed.

In [ ]:
# Install the packages in the same Python that runs the notebook
import sys
!{sys.executable} -m pip install remotezip pyarrow

**Cell: download MODIS for Brazil, 2000-2025.**
For each year, download the "all countries" zip from FIRMS, take only the Brazil CSV out of it,
delete the zip, and finally join all years in one CSV.
Result: 26 years downloaded, no failures.
Note: this notebook saved the files in `data/brazil`. Later cells read them from `~/firms_brazil`,
so the files were moved or downloaded there.

In [ ]:
import shutil
import time
import zipfile
from pathlib import Path

import pandas as pd
import requests

# {year} is replaced for each year
URL = "https://firms.modaps.eosdis.nasa.gov/data/country/zips/modis_{year}_all_countries.zip"
OUT_DIR = Path("data/brazil")
OUT_DIR.mkdir(parents=True, exist_ok=True)

YEARS = range(2000, 2026)  # 2000 to 2025


def download_brazil(year, retries=3):
    # If the year was already downloaded, reuse it
    target = OUT_DIR / f"modis_{year}_Brazil.csv"
    if target.exists():
        return target

    # Download the zip in 1 MB chunks, with up to 3 attempts (wait 5 s, 10 s... between them)
    tmp_zip = OUT_DIR / f"tmp_{year}.zip"
    for attempt in range(1, retries + 1):
        try:
            with requests.get(URL.format(year=year), stream=True, timeout=60) as r:
                r.raise_for_status()
                with open(tmp_zip, "wb") as f:
                    for chunk in r.iter_content(chunk_size=1 << 20):
                        f.write(chunk)
            break
        except requests.RequestException as e:
            print(f"{year}: attempt {attempt} failed ({e})")
            if attempt == retries:
                tmp_zip.unlink(missing_ok=True)
                raise
            time.sleep(5 * attempt)

    # Inside the zip, keep only the CSV of Brazil
    with zipfile.ZipFile(tmp_zip) as z:
        names = [n for n in z.namelist()
                 if "brazil" in n.lower() and n.lower().endswith(".csv")]
        if not names:
            tmp_zip.unlink()
            raise FileNotFoundError(f"{year}: no Brazil CSV in zip")
        with z.open(names[0]) as src, open(target, "wb") as dst:
            shutil.copyfileobj(src, dst)

    tmp_zip.unlink()
    return target


files, failed = [], []
for year in YEARS:
    try:
        files.append(download_brazil(year))
        print(f"{year}: OK")
    except Exception as e:
        failed.append(year)
        print(f"{year}: FAILED - {e}")

print("Failed years:", failed)

# Join all years into one file
df_modis = pd.concat((pd.read_csv(f) for f in files), ignore_index=True)
df_modis.to_csv(OUT_DIR / "modis_brazil_2000_2025.csv", index=False)
print(df_modis.shape)

**Cell: download VIIRS NOAA-20 (jpss1) for Brazil, 2018-2025.**
Same logic as the MODIS cell, with another URL and folder.
In this saved run the cell was stopped by hand (KeyboardInterrupt) during 2020.
Files already downloaded are skipped when you run it again.

In [ ]:
import shutil
import time
import zipfile
from pathlib import Path

import pandas as pd
import requests

URL = "https://firms.modaps.eosdis.nasa.gov/data/country/zips/viirs-jpss1_{year}_all_countries.zip"
OUT_DIR = Path("data/brazil_viirs_jpss1")
OUT_DIR.mkdir(parents=True, exist_ok=True)

YEARS = range(2018, 2026)  # 2018 to 2025


def download_brazil(year, retries=3):
    # Skip years that already exist
    target = OUT_DIR / f"viirs-jpss1_{year}_Brazil.csv"
    if target.exists():
        return target

    # Download with retries
    tmp_zip = OUT_DIR / f"tmp_{year}.zip"
    for attempt in range(1, retries + 1):
        try:
            with requests.get(URL.format(year=year), stream=True, timeout=60) as r:
                r.raise_for_status()
                with open(tmp_zip, "wb") as f:
                    for chunk in r.iter_content(chunk_size=1 << 20):
                        f.write(chunk)
            break
        except requests.RequestException as e:
            print(f"{year}: attempt {attempt} failed ({e})")
            if attempt == retries:
                tmp_zip.unlink(missing_ok=True)
                raise
            time.sleep(5 * attempt)

    # Extract only the Brazil CSV
    with zipfile.ZipFile(tmp_zip) as z:
        names = [n for n in z.namelist()
                 if "brazil" in n.lower() and n.lower().endswith(".csv")]
        if not names:
            tmp_zip.unlink()
            raise FileNotFoundError(f"{year}: no Brazil CSV in zip")
        with z.open(names[0]) as src, open(target, "wb") as dst:
            shutil.copyfileobj(src, dst)

    tmp_zip.unlink()
    return target


files, failed = [], []
for year in YEARS:
    try:
        files.append(download_brazil(year))
        print(f"{year}: OK")
    except Exception as e:
        failed.append(year)
        print(f"{year}: FAILED - {e}")

print("Failed years:", failed)

# Join all years into one file
df = pd.concat((pd.read_csv(f) for f in files), ignore_index=True)
df.to_csv(OUT_DIR / "viirs_jpss1_brazil_2018_2025.csv", index=False)
print(df.shape)

## 2. What the data is

- NASA FIRMS stores heat points detected by satellites. Each row is one detection.
- Two instruments: **MODIS** (1 km pixel, from 2000) and **VIIRS** (375 m pixel, from 2012).
- MODIS flies on two satellites: **Terra** (from 2000) and **Aqua** (from 2002).
- VIIRS flies on **S-NPP** (2012), **NOAA-20** (2018) and **NOAA-21** (2023).
- We use MODIS Terra only (no jump in 2002) and VIIRS NOAA-20 only (the one the FIRMS API gives us).

**Cell: list the MODIS files and look at the columns.**
26 files found (one per year). The first rows show one detection per row:
position, `acq_date`, `acq_time` (UTC), `satellite`, `confidence`, `frp`, `type`.

In [ ]:
import os, glob
import pandas as pd

ROOT = os.path.expanduser("~/firms_brazil")

# Recursive search: the CSVs are inside sub-folders
files = sorted(glob.glob(f"{ROOT}/modis/**/*.csv", recursive=True))
print(len(files), "MODIS files")

# check the columns of one file (first 5 rows only)
pd.read_csv(files[0], nrows=5)

## 3. Filter and aggregate (cell x week)

The raw files have millions of rows. To compare the sensors and train models we need a smaller,
standard table. We group the detections by 5 km cell and by week.
- `CELL`: cell size, 0.05 degree (about 5.5 km).
- `ORIGIN`: day 1 of the week count. Week 0 is the first week of 2000.

Filters:
- MODIS: Terra only, confidence >= 30, `type == 0`.
- VIIRS: confidence "n" (nominal) or "h" (high), `type == 0`.
- `type == 0` = presumed vegetation fire. It removes volcanoes, fixed sources and offshore sources.
- Time: UTC minus 3 hours (Brasilia time).

Result: one row per cell and week, only where fire was detected, with
`n_det` (detections), `frp_sum` (total energy) and `frp_max` (peak energy).
Output files: `pos_modis_terra.parquet` (1,759,393 rows) and `pos_viirs-jpss1.parquet` (2,409,554 rows).

In [ ]:
import os, glob, gc
import numpy as np
import pandas as pd

ROOT = os.path.expanduser("~/firms_brazil")
SAVE_DIR = os.path.expanduser("~/firms_output")
os.makedirs(SAVE_DIR, exist_ok=True)

CELL = 0.05                        # degrees (~5.5 km)
ORIGIN = pd.Timestamp("2000-01-01")

# sensor -> (folder, extra columns to read, quality filter)
CONFIG = {
    "modis_terra": dict(
        pasta="modis",
        extra=["satellite"],
        filtro=lambda d: (d["satellite"] == "Terra") & (d["confidence"] >= 30) & (d["type"] == 0),
    ),
    "viirs-jpss1": dict(
        pasta="viirs-jpss1",
        extra=[],
        filtro=lambda d: d["confidence"].isin(["n", "h"]) & (d["type"] == 0),
    ),
}

def agregar(nome, cfg):
    cols = ["latitude", "longitude", "acq_date", "acq_time", "frp", "confidence", "type"] + cfg["extra"]
    partes = []
    arquivos = sorted(glob.glob(f"{ROOT}/{cfg['pasta']}/**/*.csv", recursive=True))
    if not arquivos:
        raise FileNotFoundError(f"Nenhum CSV em {ROOT}/{cfg['pasta']}")
    for f in arquivos:
        # Read one year at a time with small data types, to save memory
        d = pd.read_csv(f, usecols=cols,
                        dtype={"latitude": "float32", "longitude": "float32",
                               "frp": "float32", "acq_time": "int16"})
        d = d[cfg["filtro"](d)].copy()          # apply the quality filter
        if d.empty:
            continue
        t = d["acq_time"].astype(int)
        # Build the date and time, then subtract 3 hours (UTC -> UTC-3)
        dt = (pd.to_datetime(d["acq_date"]) + pd.to_timedelta(t // 100, unit="h")
              + pd.to_timedelta(t % 100, unit="m") - pd.Timedelta(hours=3))
        # week = number of 7-day blocks since 2000-01-01
        d["week"] = ((dt.dt.floor("D") - ORIGIN).dt.days // 7).astype("int32")
        # cell index: latitude and longitude divided by the cell size (rounded down)
        d["cy"] = np.floor(d["latitude"] / CELL).astype("int32")
        d["cx"] = np.floor(d["longitude"] / CELL).astype("int32")
        # one row per cell and week: count, total energy, peak energy
        g = (d.groupby(["cy", "cx", "week"])
               .agg(n_det=("frp", "size"), frp_sum=("frp", "sum"), frp_max=("frp", "max"))
               .reset_index())
        partes.append(g)
        print(f"  {os.path.basename(f)}: {len(d):,} detecções -> {len(g):,} célula-semanas")
        del d, dt, t, g; gc.collect()

    # A week can cross two years, so group again over all files
    out = pd.concat(partes, ignore_index=True)
    out = (out.groupby(["cy", "cx", "week"])
              .agg(n_det=("n_det", "sum"), frp_sum=("frp_sum", "sum"), frp_max=("frp_max", "max"))
              .reset_index())
    return out

for nome, cfg in CONFIG.items():
    print(f"=== {nome} ===")
    pos = agregar(nome, cfg)
    pos.to_parquet(f"{SAVE_DIR}/pos_{nome}.parquet")
    print(f"  salvo: pos_{nome}.parquet | {pos.shape}")
    del pos; gc.collect()

## 4. Harmonization

MODIS and VIIRS count fire differently: VIIRS finds many more small fires (about 9.4 detections
for each MODIS detection). We convert VIIRS to the MODIS scale to get one continuous series.

**Cell: first attempt (replaced).** A log regression per 5 degree region.
Result: the converted VIIRS total was only **58%** of the MODIS total (see "calibrada/MODIS: 0.58").
So we replaced it with the ratio factor in the next cell. The next cell overwrites `harmonized.parquet`.

In [ ]:
import os
import numpy as np
import pandas as pd

SAVE_DIR = os.path.expanduser("~/firms_output")
ORIGIN = pd.Timestamp("2000-01-01")
CELL = 0.05
BLOCO = 100          # calibration region = 100 cells = 5 degrees
MIN_LINHAS = 500     # minimum rows in the overlap to fit one region

pos_m = pd.read_parquet(f"{SAVE_DIR}/pos_modis_terra.parquet")
pos_v = pd.read_parquet(f"{SAVE_DIR}/pos_viirs-jpss1.parquet")

# ---------- 1) Overlap: put both sensors side by side ----------
# Overlap = weeks where both sensors exist (2018-03-31 to 2025-12-27)
w_ini = int(pos_v["week"].min())
w_fim = int(min(pos_v["week"].max(), pos_m["week"].max()))
print("sobreposição:", ORIGIN + pd.Timedelta(weeks=w_ini), "->", ORIGIN + pd.Timedelta(weeks=w_fim))

chave = ["cy", "cx", "week"]
ov = (pos_m[pos_m["week"].between(w_ini, w_fim)]
      .merge(pos_v[pos_v["week"].between(w_ini, w_fim)],
             on=chave, how="outer", suffixes=("_m", "_v")))
# A cell-week that exists in one sensor only has 0 in the other
for c in ["n_det_m", "n_det_v", "frp_sum_m", "frp_sum_v"]:
    ov[c] = ov[c].fillna(0)

def regiao(df):
    # region id = 5 degree block (100 cells), written as "row_column"
    return (df["cy"] // BLOCO).astype(str) + "_" + (df["cx"] // BLOCO).astype(str)
ov["regiao"] = regiao(ov)

# ---------- 2) Calibration: log1p(MODIS) = a + b * log1p(VIIRS), per region ----------
def ajustar(df, col_v, col_m):
    # one global fit (fallback) and one fit per region with enough data
    x, y = np.log1p(df[col_v].to_numpy()), np.log1p(df[col_m].to_numpy())
    b, a = np.polyfit(x, y, 1)
    glob = (a, b)
    por_reg = {}
    for r, g in df.groupby("regiao"):
        if len(g) >= MIN_LINHAS:
            bb, aa = np.polyfit(np.log1p(g[col_v].to_numpy()), np.log1p(g[col_m].to_numpy()), 1)
            por_reg[r] = (aa, bb)
    return glob, por_reg

def aplicar(df, col, coef):
    # apply the regression of each region (or the global one) and go back from the log scale
    glob, por_reg = coef
    reg = regiao(df)
    a = reg.map(lambda r: por_reg.get(r, glob)[0]).to_numpy()
    b = reg.map(lambda r: por_reg.get(r, glob)[1]).to_numpy()
    y = np.clip(np.expm1(a + b * np.log1p(df[col].to_numpy())), 0, None)
    return np.where(df[col].to_numpy() == 0, 0.0, y)

coef_n = ajustar(ov, "n_det_v", "n_det_m")
coef_f = ajustar(ov, "frp_sum_v", "frp_sum_m")
print("ajuste global (a, b) contagem:", np.round(coef_n[0], 3), "| regiões próprias:", len(coef_n[1]))

# ---------- 3) Check: monthly totals, MODIS vs raw VIIRS vs calibrated VIIRS ----------
chk = ov.copy()
chk["n_cal"] = aplicar(chk, "n_det_v", coef_n)
chk["mes"] = (ORIGIN + pd.to_timedelta(chk["week"] * 7, unit="D")).dt.to_period("M")
mensal = chk.groupby("mes")[["n_det_m", "n_det_v", "n_cal"]].sum()
print(mensal.corr().round(3))
print("razão total VIIRS/MODIS bruta:", round(mensal.n_det_v.sum() / mensal.n_det_m.sum(), 2),
      "| calibrada/MODIS:", round(mensal.n_cal.sum() / mensal.n_det_m.sum(), 2))

# ---------- 4) Continuous series: MODIS before 2018, calibrated VIIRS after ----------
cedo = pos_m[pos_m["week"] < w_ini].copy()
cedo["sensor"] = "modis_terra"
cedo = cedo.rename(columns={"n_det": "n_harm", "frp_sum": "frp_sum_harm", "frp_max": "frp_max_harm"})

tarde = pos_v[pos_v["week"] >= w_ini].copy()
tarde["sensor"] = "viirs"
tarde["n_harm"] = aplicar(tarde, "n_det", coef_n)
tarde["frp_sum_harm"] = aplicar(tarde, "frp_sum", coef_f)
tarde["frp_max_harm"] = tarde["frp_max"]          # peak FRP is not calibrated
tarde = tarde.drop(columns=["n_det", "frp_sum", "frp_max"])

cols = ["cy", "cx", "week", "n_harm", "frp_sum_harm", "frp_max_harm", "sensor"]
harm = pd.concat([cedo[cols], tarde[cols]], ignore_index=True)
harm["week_start"] = ORIGIN + pd.to_timedelta(harm["week"] * 7, unit="D")
# cell center (latitude and longitude) from the cell index
harm["lat"] = (harm["cy"] + 0.5) * CELL
harm["lon"] = (harm["cx"] + 0.5) * CELL
harm.to_parquet(f"{SAVE_DIR}/harmonized.parquet")
print(harm.groupby("sensor").size(), harm.shape)

**Cell: final harmonization (ratio factor).**
For each 5 degree region and calendar month: factor = total MODIS / total VIIRS, on the overlap years.
We multiply VIIRS by this factor.
- Validation: fit on 2018-2022, test on 2023-2025. Result: converted VIIRS = **93%** of MODIS,
  monthly correlation **0.986**.
- Final fit on all overlap years: global factor 0.106 for counts and 0.348 for FRP.
- The last print shows monthly totals around the change from MODIS to VIIRS (2017 to 2018): the series is continuous.
- This cell uses `ov`, `regiao`, `pos_m`, `pos_v`, `w_ini` from the previous cell. Run that cell first.

In [ ]:
ov["mes_ano"] = (ORIGIN + pd.to_timedelta(ov["week"] * 7, unit="D")).dt.month

def fatores(df, col_v, col_m, min_v=50):
    # sum by region and month, then divide MODIS by VIIRS
    g = df.groupby(["regiao", "mes_ano"])[[col_v, col_m]].sum()
    glob = df[col_m].sum() / df[col_v].sum()
    f = (g[col_m] / g[col_v]).where(g[col_v] >= min_v)   # avoids an unstable factor with little data
    return f.fillna(glob), glob                          # global factor as fallback

def aplicar_f(df, col, f, glob):
    # look up the factor of each row (region + month) and multiply
    d = pd.DataFrame({"regiao": regiao(df).to_numpy(),
                      "mes_ano": (ORIGIN + pd.to_timedelta(df["week"].to_numpy() * 7, unit="D")).month})
    d = d.merge(f.rename("f").reset_index(), on=["regiao", "mes_ano"], how="left")
    return df[col].to_numpy() * d["f"].fillna(glob).to_numpy()

# ---------- Temporal validation: fit on 2018-2022, test on 2023-2025 ----------
w_corte = int((pd.Timestamp("2023-01-01") - ORIGIN).days // 7)
treino, teste = ov[ov["week"] < w_corte], ov[ov["week"] >= w_corte].copy()
f_tr, g_tr = fatores(treino, "n_det_v", "n_det_m")
teste["n_cal"] = aplicar_f(teste, "n_det_v", f_tr, g_tr)
teste["mes"] = (ORIGIN + pd.to_timedelta(teste["week"] * 7, unit="D")).dt.to_period("M")
m = teste.groupby("mes")[["n_det_m", "n_cal"]].sum()
print("TESTE 2023-2025 | razão calibrada/MODIS:", round(m.n_cal.sum() / m.n_det_m.sum(), 2),
      "| correlação mensal:", round(m.corr().iloc[0, 1], 3))

# ---------- Final fit with the whole overlap ----------
f_n, g_n = fatores(ov, "n_det_v", "n_det_m")
f_f, g_f = fatores(ov, "frp_sum_v", "frp_sum_m")
print("fator global contagem:", round(g_n, 3), "| FRP:", round(g_f, 3))

# ---------- Continuous series ----------
cedo = pos_m[pos_m["week"] < w_ini].copy()
cedo["sensor"] = "modis_terra"
cedo = cedo.rename(columns={"n_det": "n_harm", "frp_sum": "frp_sum_harm", "frp_max": "frp_max_harm"})

tarde = pos_v[pos_v["week"] >= w_ini].copy()
tarde["sensor"] = "viirs"
tarde["n_harm"] = aplicar_f(tarde, "n_det", f_n, g_n)
tarde["frp_sum_harm"] = aplicar_f(tarde, "frp_sum", f_f, g_f)
tarde["frp_max_harm"] = tarde["frp_max"]
tarde = tarde.drop(columns=["n_det", "frp_sum", "frp_max"])

cols = ["cy", "cx", "week", "n_harm", "frp_sum_harm", "frp_max_harm", "sensor"]
harm = pd.concat([cedo[cols], tarde[cols]], ignore_index=True)
harm["week_start"] = ORIGIN + pd.to_timedelta(harm["week"] * 7, unit="D")
harm["lat"] = (harm["cy"] + 0.5) * CELL
harm["lon"] = (harm["cx"] + 0.5) * CELL
harm.to_parquet(f"{SAVE_DIR}/harmonized.parquet")

# check: the monthly total of the series must be continuous at the 2017 -> 2018 transition
tot = harm.groupby(harm["week_start"].dt.to_period("M"))["n_harm"].sum()
print(tot["2017-01":"2018-12"].round(0).to_string())

## 5. Training table

One row = one cell in one target week. Each row has:
- **Label:** `y_pres` (fire in the target week: 1 or 0) and `y_int` (log of the total FRP, only where there was fire).
- **Features:** the last 4 weeks of the cell (`n_l0..3`, `frp_l0..3`), the last 4 weeks of the 8 neighbors
  (`nb_l0..3`), the history of the same week in earlier years (`clim_n`), season (`doy_sin`, `doy_cos`),
  position (`lat`, `lon`).
- **Weight** (`peso`): we sample only 2 "no fire" rows for each "fire" row. The weight gives back the real fire rate.
- Temporal split: train before 2023, test from 2023.

Result: 3,667,483 positives, 7,334,966 negatives, real fire rate 1.35% (all years).
Baselines to beat (test AUC): `clim_n` 0.704 and `n_l0` 0.619.

In [ ]:
import os, gc
import numpy as np
import pandas as pd

SAVE_DIR = os.path.expanduser("~/firms_output")
ORIGIN = pd.Timestamp("2000-01-01")
CELL = 0.05
HORIZONTE = 1        # weeks ahead (change to 4 later)
N_LAGS = 4           # reference weeks: t, t-1, t-2, t-3
NEG_RATIO = 2        # sampled negatives per positive
ANO_TESTE = 2023     # train: before 2023 | test: 2023 onwards
SEED = 42

harm = pd.read_parquet(f"{SAVE_DIR}/harmonized.parquet",
                       columns=["cy", "cx", "week", "n_harm", "frp_sum_harm", "sensor"])
harm = harm.astype({"cy": "int32", "cx": "int32", "week": "int32",
                    "n_harm": "float32", "frp_sum_harm": "float32"})
W_VIIRS = int(harm.loc[harm["sensor"] == "viirs", "week"].min())   # first VIIRS week
harm = harm.drop(columns="sensor")
w_min, w_max = int(harm["week"].min()), int(harm["week"].max())
WA_MIN = w_min + HORIZONTE + N_LAGS - 1      # first target week that has all lags

# ---------- 1) Samples: positives (fire in the target week) and sampled negatives ----------
rng = np.random.default_rng(SEED)
pos = (harm[harm["week"] >= WA_MIN]
       .rename(columns={"week": "week_alvo"})[["cy", "cx", "week_alvo", "frp_sum_harm"]]
       .reset_index(drop=True))
pos["y_pres"] = 1

# Negatives: random cell + random week, then drop the pairs that really had fire
cells = harm[["cy", "cx"]].drop_duplicates().to_numpy()
n_universo = len(cells) * (w_max - WA_MIN + 1)      # all possible cell-weeks
n_neg = NEG_RATIO * len(pos)
m = int(n_neg * 1.2)                                # draw 20% extra, because some will be removed
idx = rng.integers(0, len(cells), m)
neg = pd.DataFrame({"cy": cells[idx, 0].astype("int32"),
                    "cx": cells[idx, 1].astype("int32"),
                    "week_alvo": rng.integers(WA_MIN, w_max + 1, m).astype("int32")}).drop_duplicates()
neg = neg.merge(pos[["cy", "cx", "week_alvo"]], how="left", indicator=True)
neg = neg[neg["_merge"] == "left_only"].drop(columns="_merge").head(n_neg)
neg["frp_sum_harm"] = np.float32(np.nan)
neg["y_pres"] = 0

base = pd.concat([pos, neg], ignore_index=True)
base["y_int"] = np.log1p(base["frp_sum_harm"]).astype("float32")   # NaN for negatives
base = base.drop(columns="frp_sum_harm")
# weight: restores the real fire rate (the sampled negatives represent the whole universe)
base["peso"] = np.where(base["y_pres"] == 1, 1.0,
                        (n_universo - len(pos)) / len(neg)).astype("float32")
print(f"positivos: {len(pos):,} | negativos: {len(neg):,} | taxa real de fogo: {len(pos)/n_universo:.5f}")
del pos, neg; gc.collect()

# ---------- 2) Neighborhood: sum of the 8 neighbor cells, per week ----------
# Shift the table by one cell in each of the 8 directions, then add the values that land on the same cell
partes = []
for dy in (-1, 0, 1):
    for dx in (-1, 0, 1):
        if (dy, dx) != (0, 0):
            p = harm[["cy", "cx", "week", "n_harm"]].copy()
            p["cy"] += dy
            p["cx"] += dx
            partes.append(p)
nb = pd.concat(partes, ignore_index=True).groupby(["cy", "cx", "week"], as_index=False)["n_harm"].sum()
del partes; gc.collect()

# ---------- 3) Lags: activity in (target week - HORIZON - k); 0 if there was no detection ----------
def lag_tbl(df, col, nome, k):
    t = df[["cy", "cx", "week", col]].copy()
    t["week_alvo"] = (t["week"] + HORIZONTE + k).astype("int32")   # move the week forward to the target week
    return t.drop(columns="week").rename(columns={col: f"{nome}_l{k}"})

feat_lags = []
for nome, tab, col in [("n", harm, "n_harm"), ("frp", harm, "frp_sum_harm"), ("nb", nb, "n_harm")]:
    for k in range(N_LAGS):
        base = base.merge(lag_tbl(tab, col, nome, k), on=["cy", "cx", "week_alvo"], how="left")
        base[f"{nome}_l{k}"] = base[f"{nome}_l{k}"].fillna(0).astype("float32")
        feat_lags.append(f"{nome}_l{k}")
del nb; gc.collect()

# ---------- 4) Calendar, position and sensor of the target week ----------
ws = ORIGIN + pd.to_timedelta(base["week_alvo"].to_numpy() * 7, unit="D")
doy = ws.dayofyear.to_numpy()
base["ano"] = ws.year.to_numpy().astype("int16")
base["woy"] = ((doy - 1) // 7).astype("int8")                       # week of the year (0-52)
# sin and cos make December "close" to January
base["doy_sin"] = np.sin(2 * np.pi * doy / 365.25).astype("float32")
base["doy_cos"] = np.cos(2 * np.pi * doy / 365.25).astype("float32")
base["lat"] = ((base["cy"] + 0.5) * CELL).astype("float32")
base["lon"] = ((base["cx"] + 0.5) * CELL).astype("float32")
base["is_viirs"] = (base["week_alvo"] >= W_VIIRS).astype("int8")    # 1 = VIIRS era

# ---------- 5) Cell climatology: historical mean for the same week of the year,
#               using ONLY years before the target week's year (no leakage) ----------
ws_h = ORIGIN + pd.to_timedelta(harm["week"].to_numpy() * 7, unit="D")
h = harm[["cy", "cx", "n_harm"]].copy()
h["woy"] = ((ws_h.dayofyear.to_numpy() - 1) // 7).astype("int8")
h["ano"] = ws_h.year.to_numpy().astype("int16")
ann = (h.groupby(["cy", "cx", "woy", "ano"], as_index=False)["n_harm"].sum()
         .sort_values("ano", kind="stable"))
ann["cum"] = ann.groupby(["cy", "cx", "woy"])["n_harm"].cumsum()   # running sum over the years
del h; gc.collect()

base = base.sort_values("ano", kind="stable")
# merge_asof with allow_exact_matches=False takes the last row of an EARLIER year
base = pd.merge_asof(base, ann[["cy", "cx", "woy", "ano", "cum"]], on="ano",
                     by=["cy", "cx", "woy"], allow_exact_matches=False, direction="backward")
anos_antes = (base["ano"].astype("int32") - 2000).clip(lower=1)    # number of earlier years
base["clim_n"] = (base["cum"].fillna(0) / anos_antes).astype("float32")
base = base.drop(columns=["cum", "woy"]).reset_index(drop=True)
del ann; gc.collect()

# ---------- 6) Time split and saving ----------
base["split"] = np.where(base["ano"] >= ANO_TESTE, "teste", "treino")
FEATURES = feat_lags + ["clim_n", "doy_sin", "doy_cos", "lat", "lon", "is_viirs"]
base.to_parquet(f"{SAVE_DIR}/treino_h{HORIZONTE}.parquet")

print(base.groupby("split").agg(linhas=("y_pres", "size"), taxa_na_amostra=("y_pres", "mean")))
print("features:", FEATURES)

# baselines to beat (weighted AUC on the test set)
try:
    from sklearn.metrics import roc_auc_score
    te = base[base["split"] == "teste"]
    for nome in ["clim_n", "n_l0"]:
        print(f"AUC baseline {nome}:", round(roc_auc_score(te["y_pres"], te[nome], sample_weight=te["peso"]), 3))
except ImportError:
    pass

**Cell: sensor-era diagnostic.**
The fire rate depends on the sensor era: **0.67%** in the MODIS era (`is_viirs=0`) and about **3%**
in the VIIRS era (`is_viirs=1`). So the labels of the two eras are not comparable.
Decision: train on the VIIRS era only (2018-2022), and test on 2023-2025.

In [ ]:
from sklearn.metrics import roc_auc_score

base = pd.read_parquet(f"{SAVE_DIR}/treino_h1.parquet")

def taxa_pond(d):
    # fire rate with the weights (the real rate)
    return (d["y_pres"] * d["peso"]).sum() / d["peso"].sum()

for (viirs, split), d in base.groupby(["is_viirs", "split"]):
    print(f"is_viirs={viirs} | {split:6s} | linhas={len(d):>9,} | taxa ponderada={taxa_pond(d):.4f} | "
          f"AUC clim={roc_auc_score(d['y_pres'], d['clim_n'], sample_weight=d['peso']):.3f} | "
          f"AUC n_l0={roc_auc_score(d['y_pres'], d['n_l0'], sample_weight=d['peso']):.3f}")

**Cell: final training table (VIIRS era only) with two more climatology features.**
- `clim_n3`: same as `clim_n`, but with a window of +-1 week (less noisy).
- `clim_reg`: climatology of the whole 5 degree region (regional context).
Output: `treino_h1_viirs.parquet`.
Baseline AUCs on the test set: `clim_n` 0.704, `clim_n3` 0.755, `clim_reg` 0.72, `n_l0` 0.619.
The best simple baseline is `clim_n3` (0.755), so a model must beat that number.

In [ ]:
import os, gc
import numpy as np
import pandas as pd

SAVE_DIR = os.path.expanduser("~/firms_output")
ORIGIN = pd.Timestamp("2000-01-01")
HORIZONTE = 1
BLOCO = 100            # regional block = 100 cells = 5 degrees

base = pd.read_parquet(f"{SAVE_DIR}/treino_h{HORIZONTE}.parquet")
base = base[base["is_viirs"] == 1].reset_index(drop=True)      # keep the VIIRS era only

def calendario(semanas):
    # week of the year (0-52) and year, from the week number
    ws = ORIGIN + pd.to_timedelta(np.asarray(semanas) * 7, unit="D")
    return ((ws.dayofyear.to_numpy() - 1) // 7).astype("int8"), ws.year.to_numpy().astype("int16")

base["woy"], _ = calendario(base["week_alvo"])
base["ry"] = (base["cy"] // BLOCO).astype("int16")
base["rx"] = (base["cx"] // BLOCO).astype("int16")

harm = pd.read_parquet(f"{SAVE_DIR}/harmonized.parquet", columns=["cy", "cx", "week", "n_harm"])
harm = harm.astype({"cy": "int32", "cx": "int32", "week": "int32", "n_harm": "float32"})
harm["woy"], harm["ano"] = calendario(harm["week"])
harm["ry"] = (harm["cy"] // BLOCO).astype("int16")
harm["rx"] = (harm["cx"] // BLOCO).astype("int16")

# annual tables: cell with a +-1 week window, and region (5 degree block)
partes = []
for d in (-1, 0, 1):
    p = harm[["cy", "cx", "woy", "ano", "n_harm"]].copy()
    p["woy"] = ((p["woy"].astype("int16") + d) % 53).astype("int8")   # shift the week of the year by -1, 0, +1
    partes.append(p)
cel3 = (pd.concat(partes, ignore_index=True)
          .groupby(["cy", "cx", "woy", "ano"], as_index=False)["n_harm"].sum())
tab_reg = harm.groupby(["ry", "rx", "woy", "ano"], as_index=False)["n_harm"].sum()
del partes, harm; gc.collect()

def clim_anterior(base, tab, by, nome):
    """Historical mean for the same time of year, using only years before the target week's year."""
    tab = tab.sort_values("ano", kind="stable").copy()
    tab["cum"] = tab.groupby(by)["n_harm"].cumsum()
    b = base.sort_values("ano", kind="stable")
    b = pd.merge_asof(b, tab[by + ["ano", "cum"]], on="ano", by=by,
                      allow_exact_matches=False, direction="backward")
    anos = (b["ano"].astype("int32") - 2000).clip(lower=1)
    b[nome] = (b["cum"].fillna(0) / anos).astype("float32")
    return b.drop(columns="cum").reset_index(drop=True)

base = clim_anterior(base, cel3, ["cy", "cx", "woy"], "clim_n3")
base = clim_anterior(base, tab_reg, ["ry", "rx", "woy"], "clim_reg")
base = base.drop(columns=["woy", "ry", "rx"])
base.to_parquet(f"{SAVE_DIR}/treino_h{HORIZONTE}_viirs.parquet")

from sklearn.metrics import roc_auc_score
te = base[base["split"] == "teste"]
for c in ["clim_n", "clim_n3", "clim_reg", "n_l0"]:
    print(f"AUC baseline {c:9s} (teste):", round(roc_auc_score(te["y_pres"], te[c], sample_weight=te["peso"]), 3))

## 6. Models

**Cell: train the two chosen models (HistGradientBoosting).**
- Presence model (classifier): probability of at least one detection in the cell next week.
- Intensity model (regressor): log1p of the total FRP, trained **only on rows with fire**.
- Both use 19 features (the `is_viirs` feature is not used, because the table is VIIRS era only).

Results on the test set (2023-2025):
- Presence: AUC **0.866** (baseline `clim_n3` 0.755), average precision 0.253 (random: 0.027),
  log-loss 0.095 (constant: 0.1257), Brier 0.0232 (constant: 0.0267).
- The most useful features (permutation importance): `clim_n3`, `nb_l0`, `lat`, `n_l0`.
- Intensity: MAE 0.785 (constant mean: 0.965), RMSE 1.037, R2 **0.29**. Intensity is hard to predict.
- The models are saved in `modelos_h1.joblib`.

In [ ]:
import joblib
from sklearn.ensemble import HistGradientBoostingClassifier, HistGradientBoostingRegressor
from sklearn.metrics import (roc_auc_score, average_precision_score, log_loss, brier_score_loss,
                             mean_absolute_error, mean_squared_error, r2_score)
from sklearn.inspection import permutation_importance

base = pd.read_parquet(f"{SAVE_DIR}/treino_h{HORIZONTE}_viirs.parquet")
# 19 features: 12 lags (cell, energy, neighbors) + 3 climatologies + season + position
FEATURES = ([f"{n}_l{k}" for n in ("n", "frp", "nb") for k in range(4)]
            + ["clim_n", "clim_n3", "clim_reg", "doy_sin", "doy_cos", "lat", "lon"])
tr, te = base[base["split"] == "treino"], base[base["split"] == "teste"]

# ---------- Presence model ----------
clf = HistGradientBoostingClassifier(learning_rate=0.1, max_iter=300, max_leaf_nodes=63,
                                     min_samples_leaf=200, l2_regularization=1.0,
                                     early_stopping=False, random_state=42)
clf.fit(tr[FEATURES], tr["y_pres"], sample_weight=tr["peso"])   # the weights restore the real fire rate
p = clf.predict_proba(te[FEATURES])[:, 1]                       # probability of fire
w, y = te["peso"], te["y_pres"]
p0 = np.average(tr["y_pres"], weights=tr["peso"])          # constant rate (reference)

print("=== PRESENÇA (teste 2023-2025, ponderado) ===")
print("AUC            :", round(roc_auc_score(y, p, sample_weight=w), 3), "| clim_n3:", round(roc_auc_score(y, te["clim_n3"], sample_weight=w), 3))
print("AUC-PR         :", round(average_precision_score(y, p, sample_weight=w), 3), "| taxa de fogo (piso):", round(np.average(y, weights=w), 3))
print("log-loss       :", round(log_loss(y, p, sample_weight=w), 4), "| constante:", round(log_loss(y, np.full(len(y), p0), sample_weight=w), 4))
print("Brier          :", round(brier_score_loss(y, p, sample_weight=w), 4), "| constante:", round(brier_score_loss(y, np.full(len(y), p0), sample_weight=w), 4))

# Permutation importance: how much the AUC drops when one feature is shuffled (100,000-row sample)
sub = te.sample(100_000, random_state=0)
pi = permutation_importance(clf, sub[FEATURES], sub["y_pres"], scoring="roc_auc", n_repeats=3,
                            random_state=0, sample_weight=sub["peso"])
print(pd.Series(pi.importances_mean, index=FEATURES).sort_values(ascending=False).round(4).head(10))

# ---------- Intensity model (only where there was fire) ----------
tr_i, te_i = tr[tr["y_pres"] == 1], te[te["y_pres"] == 1]
reg = HistGradientBoostingRegressor(learning_rate=0.1, max_iter=300, max_leaf_nodes=31,
                                    min_samples_leaf=100, l2_regularization=1.0,
                                    early_stopping=False, random_state=42)
reg.fit(tr_i[FEATURES], tr_i["y_int"])
pi_ = reg.predict(te_i[FEATURES])
base_i = np.full(len(te_i), tr_i["y_int"].mean())      # reference: always predict the mean

print("\n=== INTENSIDADE (log1p do FRP somado, teste) ===")
print("MAE :", round(mean_absolute_error(te_i["y_int"], pi_), 3), "| média constante:", round(mean_absolute_error(te_i["y_int"], base_i), 3))
print("RMSE:", round(np.sqrt(mean_squared_error(te_i["y_int"], pi_)), 3), "| média constante:", round(np.sqrt(mean_squared_error(te_i["y_int"], base_i)), 3))
print("R²  :", round(r2_score(te_i["y_int"], pi_), 3))

# Save both models and the feature list in one file
joblib.dump({"presenca": clf, "intensidade": reg, "features": FEATURES}, f"{SAVE_DIR}/modelos_h{HORIZONTE}.joblib")

**Cell: Naive Bayes test.** A simple probability model, tried for comparison.
Result: AUC **0.799** (HistGB: 0.866). It is fast but worse.
This cell uses `tr`, `te`, `FEATURES`, `y`, `w` from the previous cell.

In [ ]:
from sklearn.naive_bayes import GaussianNB

nb = GaussianNB()
nb.fit(tr[FEATURES], tr["y_pres"], sample_weight=tr["peso"])
p_nb = nb.predict_proba(te[FEATURES])[:, 1]
print("AUC Naive Bayes:", round(roc_auc_score(y, p_nb, sample_weight=w), 3))

**Cell: compare more models.**
- Train on 2018-2021, validate on 2022, test on 2023-2025.
- Counts and FRP are very skewed, so the features go through `log1p` (helps LogReg and Naive Bayes).
- RandomForest, ExtraTrees and HistGB are commented out because they were too slow here.

| Model | AUC val | AUC test | AP test | Time |
|---|---|---|---|---|
| XGBoost | 0.865 | 0.864 | 0.240 | 90 s |
| Naive Bayes | 0.830 | 0.823 | 0.145 | 3 s |
| LogReg | 0.823 | 0.814 | 0.194 | 7 s |
| Baseline `clim_n3` | 0.759 | 0.755 | - | - |

HistGradientBoosting (0.866 in the previous cell) is as good as XGBoost, so we keep HistGB.

In [ ]:
import time
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, HistGradientBoostingClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, average_precision_score

SEM_LOG = ["doy_sin", "doy_cos", "lat", "lon"]     # these features are not skewed: no log
def prep(d):
    X = d[FEATURES].copy()
    for c in FEATURES:
        if c not in SEM_LOG:
            X[c] = np.log1p(X[c])      # counts/FRP are very skewed (helps LogReg and NB)
    return X

trn = tr[tr["ano"] < 2022]             # training data (2018-2021)
val = tr[tr["ano"] == 2022]            # validation year
Xtr, Xva, Xte = prep(trn), prep(val), prep(te)

modelos = {
    "LogReg":     make_pipeline(StandardScaler(), LogisticRegression(max_iter=300)),
    "NaiveBayes": GaussianNB(),
    # "RandForest": RandomForestClassifier(n_estimators=150, max_depth=18, min_samples_leaf=50, n_jobs=-1, random_state=0),
    # "ExtraTrees": ExtraTreesClassifier(n_estimators=150, max_depth=18, min_samples_leaf=50, n_jobs=-1, random_state=0),
    # "HistGB":     HistGradientBoostingClassifier(learning_rate=0.1, max_iter=300, max_leaf_nodes=63,
                                                 # min_samples_leaf=200, early_stopping=False, random_state=0),
}
# LightGBM and XGBoost are optional: use them only if the package is installed
try:
    from lightgbm import LGBMClassifier
    modelos["LightGBM"] = LGBMClassifier(n_estimators=400, learning_rate=0.05, num_leaves=63, verbose=-1)
except ImportError:
    pass
try:
    from xgboost import XGBClassifier
    modelos["XGBoost"] = XGBClassifier(n_estimators=400, max_depth=8, learning_rate=0.05, tree_method="hist")
except ImportError:
    pass

linhas = []
for nome, m in modelos.items():
    t0 = time.time()
    if nome == "LogReg":
        # in a pipeline, the weight argument needs the step name as prefix
        m.fit(Xtr, trn["y_pres"], logisticregression__sample_weight=trn["peso"])
    else:
        m.fit(Xtr, trn["y_pres"], sample_weight=trn["peso"])
    pv, pt = m.predict_proba(Xva)[:, 1], m.predict_proba(Xte)[:, 1]
    linhas.append({
        "modelo": nome,
        "AUC_val":  roc_auc_score(val["y_pres"], pv, sample_weight=val["peso"]),
        "AUC_teste": roc_auc_score(te["y_pres"], pt, sample_weight=te["peso"]),
        "AP_teste":  average_precision_score(te["y_pres"], pt, sample_weight=te["peso"]),
        "seg": time.time() - t0,
    })
    print(nome, "ok", round(time.time() - t0), "s")

res = pd.DataFrame(linhas).sort_values("AUC_val", ascending=False).round(3)
print(res.to_string(index=False))
print("baseline clim_n3 | AUC_val:", round(roc_auc_score(val["y_pres"], val["clim_n3"], sample_weight=val["peso"]), 3),
      "| AUC_teste:", round(roc_auc_score(te["y_pres"], te["clim_n3"], sample_weight=te["peso"]), 3))

## 7. Files for the API

**Cell: save the calibration** (the conversion factors) in `calibracao.joblib`.
The API needs them to convert live VIIRS detections to the MODIS scale.
`f_n`, `f_f`: factor tables by region and month (counts and FRP). `g_n` = 0.106 and `g_f` = 0.348:
global fallback factors. `w_ini` = 952: first week of the overlap (2018-03-31).

In [ ]:
import os, joblib

SAVE_DIR = os.path.expanduser("~/firms_output")

# check that the variables exist before saving
print(type(f_n).__name__, round(g_n, 3), type(f_f).__name__, round(g_f, 3), w_ini)

joblib.dump({"f_n": f_n, "g_n": g_n, "f_f": f_f, "g_f": g_f, "w_ini": w_ini},
            f"{SAVE_DIR}/calibracao.joblib")
print("salvo em", f"{SAVE_DIR}/calibracao.joblib")

**Cell: export the prediction package.**
`exportar` (from `exportar_artefatos.py`) writes the `./artefatos` folder: the models, the calibration,
the climatology tables (built from years before the target year 2026), the cell list, the intensity
thresholds (terciles) and `config.json`. The scikit-learn version must be the same on the API machine (1.7.2).

In [ ]:
from exportar_artefatos import exportar

cfg = exportar("~/firms_output", "./artefatos", ano_alvo=2026)
print(cfg)

**Cell: check the prediction code against the training table.**
Pick one test row with fire, rebuild its 19 features from the raw detections with `PrevisorFogo._features`,
and compare with the training table. Result: **maximum difference 0.0** for all 19 features,
so the API computes the same features as the training.

In [ ]:
import os, glob
import numpy as np
import pandas as pd
from exportar_artefatos import exportar
from previsao_fogo import PrevisorFogo

SAVE_DIR = os.path.expanduser("~/firms_output")
ROOT = os.path.expanduser("~/firms_brazil")
ORIGIN = pd.Timestamp("2000-01-01")

tab = pd.read_parquet(f"{SAVE_DIR}/treino_h1_viirs.parquet")
doy = (ORIGIN + pd.to_timedelta(tab["week_alvo"] * 7, unit="D")).dt.dayofyear
# a test row with fire, recent activity in the cell and neighbors, between day 120 and 300 of the year
cand = tab[(tab["split"] == "teste") & (tab["y_pres"] == 1) & (tab["n_l0"] > 0)
           & (tab["nb_l0"] > 0) & doy.between(120, 300)]
lin = cand.sample(1, random_state=1).iloc[0]

cy, cx, wa, ano = int(lin["cy"]), int(lin["cx"]), int(lin["week_alvo"]), int(lin["ano"])
data_ref = ORIGIN + pd.Timedelta(days=7 * (wa - 1) + 6)      # last day of the reference week
lat, lon = (cy + 0.5) * 0.05, (cx + 0.5) * 0.05
print("célula", (cy, cx), "| semana-alvo", wa, "| data_ref", data_ref.date())

# raw jpss1 detections of the year, close to the cell
arq = glob.glob(f"{ROOT}/viirs-jpss1/{ano}/**/*.csv", recursive=True)[0]
det = pd.read_csv(arq, usecols=["latitude", "longitude", "acq_date", "acq_time", "frp", "confidence", "type"])
det = det[(det["latitude"].sub(lat).abs() < 0.2) & (det["longitude"].sub(lon).abs() < 0.2)]

# artifacts with the climatology of the row's year (same as in training)
exportar("~/firms_output", "./artefatos_val", ano_alvo=ano)
pr = PrevisorFogo("./artefatos_val")
X, avisos, _, _ = pr._features(det, lat, lon, 15.0, data_ref)
x = X[(X["cy"] == cy) & (X["cx"] == cx)].iloc[0]

# compare the features computed by the function with the training table
comp = pd.DataFrame({"funcao": x[pr.features].astype(float), "treino": lin[pr.features].astype(float)})
comp["dif"] = (comp["funcao"] - comp["treino"]).abs()
print(comp.round(4).to_string())
print("avisos:", avisos, "| diferença máxima:", float(comp["dif"].max()))

**Cell: zip the package** (`artefatos` folder and `previsao_fogo.py`) to send to the back-end teammate.
`!` runs a shell command.

In [ ]:
!cd ~ && zip -r pacote_previsao.zip artefatos previsao_fogo.py

## 8. Look at the tables and the predictions

**Cell: show the first rows of the four main tables.**
- `pos_m`: MODIS Terra by cell and week (1,759,393 rows).
- `pos_v`: VIIRS NOAA-20 by cell and week (2,409,554 rows).
- `harm`: harmonized series 2000-2025 (3,671,817 rows).
- `treino`: training table (VIIRS era, 28 columns).

In [ ]:
import os
import pandas as pd

SAVE_DIR = os.path.expanduser("~/firms_output")

# Step 2: detections aggregated by cell and week
pos_m = pd.read_parquet(f"{SAVE_DIR}/pos_modis_terra.parquet")
pos_v = pd.read_parquet(f"{SAVE_DIR}/pos_viirs-jpss1.parquet")

# Step 4: harmonized series (MODIS + calibrated VIIRS)
harm = pd.read_parquet(f"{SAVE_DIR}/harmonized.parquet")

# Step 5: training table
treino = pd.read_parquet(f"{SAVE_DIR}/treino_h1_viirs.parquet")

for nome, df in [("pos_m (MODIS)", pos_m), ("pos_v (VIIRS)", pos_v),
                 ("harm", harm), ("treino", treino)]:
    print(f"\n=== {nome} | {df.shape[0]:,} linhas x {df.shape[1]} colunas ===")
    display(df.head())

**Cell: predictions on the test set.** Load the two models and predict for all test rows:
`p_fogo` = probability of fire, `frp_prev` = predicted energy (back from the log scale),
`frp_real` = real energy (only where there was fire).

In [ ]:
import os, joblib
import numpy as np
import pandas as pd

SAVE_DIR = os.path.expanduser("~/firms_output")
ORIGIN = pd.Timestamp("2000-01-01")

m = joblib.load(f"{SAVE_DIR}/modelos_h1.joblib")
clf, reg, FEATURES = m["presenca"], m["intensidade"], m["features"]

tab = pd.read_parquet(f"{SAVE_DIR}/treino_h1_viirs.parquet")
te = tab[tab["split"] == "teste"].copy()

te["p_fogo"] = clf.predict_proba(te[FEATURES])[:, 1]          # probability of fire in the target week
te["frp_prev"] = np.expm1(reg.predict(te[FEATURES]))           # predicted energy (back from log1p)
te["frp_real"] = np.expm1(te["y_int"])                         # exists only where there was fire
te["semana"] = ORIGIN + pd.to_timedelta(te["week_alvo"] * 7, unit="D")

**Cell: look at examples.** Highest probabilities (all had fire, in the dry season Aug-Sep),
lowest probabilities (no fire; cells in the north in the rainy season) and random rows.
Note: the predicted energy for a row with a very low `p_fogo` is not meaningful, because
the intensity model was trained only where there was fire.

In [ ]:
cols = ["semana", "lat", "lon", "n_l0", "nb_l0", "clim_n3", "p_fogo", "frp_prev", "y_pres", "frp_real"]

print("Maiores probabilidades de fogo:")
display(te.sort_values("p_fogo", ascending=False)[cols].head(10))

print("Menores probabilidades:")
display(te.sort_values("p_fogo")[cols].head(10))

print("Exemplos aleatórios:")
display(te.sample(10, random_state=1)[cols])

## 9. Checks of the website numbers (fire-days)

- **Fire-day** = number of distinct 0.01 degree cells (about 1 km) with at least one detection, counted per day and summed.
  It is not the same as the number of detections.
- Roraima is a rectangle here, as on the site (it is not the official state border).

**Cell: first Roraima check (has a bug).**
The folder is fixed to `2025`, but `MES` is `2024-03`. The 2025 file has no rows of March 2024,
so all three numbers are 0. The next cell fixes it.

In [ ]:
import os, glob
import pandas as pd

ROOT = os.path.expanduser("~/firms_brazil")

# Roraima as the site defines it: a rectangle, not the official state border
W, S, E, N = -64.8, -1.5, -59.0, 5.2
MES = "2024-03"

def ler(pasta, extra=()):
    arqs = glob.glob(f"{ROOT}/{pasta}/2025/**/*.csv", recursive=True)      # BUG: fixed year 2025
    cols = ["latitude", "longitude", "acq_date", "acq_time", "confidence", "type", *extra]
    d = pd.concat([pd.read_csv(f, usecols=cols) for f in arqs], ignore_index=True)
    d = d[d["acq_date"].astype(str).str.startswith(MES)]
    d = d[d["latitude"].between(S, N) & d["longitude"].between(W, E)]
    return d[d["type"] == 0]                      # vegetation fire only

# VIIRS NOAA-20: nominal or high confidence
v = ler("viirs-jpss1")
v = v[v["confidence"].isin(["n", "h"])]
v = v.drop_duplicates(["latitude", "longitude", "acq_date", "acq_time"])

# MODIS Terra: confidence of 30 or more
m = ler("modis", extra=["satellite"])
m = m[(m["satellite"] == "Terra") & (m["confidence"] >= 30)]
m = m.drop_duplicates(["latitude", "longitude", "acq_date", "acq_time"])

print("Focos VIIRS NOAA-20:", len(v))
print("Focos MODIS Terra  :", len(m))

# Site fire-days (0.01 degree cell per day): VIIRS only
v = v.assign(cy=(v["latitude"] // 0.01).astype(int),
             cx=(v["longitude"] // 0.01).astype(int))
print("Fire-days VIIRS    :", v.drop_duplicates(["cy", "cx", "acq_date"]).shape[0])

**Cell: Roraima fire-days for one month (fixed).** The folder year now comes from `MES`.
Result for March 2024 (raw NOAA-20, whole rectangle): 17,096 detections and **8,844 fire-days**.
The last table gives fire-days per day (the peak is 2024-03-22, with 1,469).

In [ ]:
import glob, os
import pandas as pd

PASTA = os.path.expanduser("~/firms_brazil/viirs-jpss1")
MES = "2024-03"                      # change here: "AAAA-MM"
W, S, E, N = -64.8, -1.5, -59.0, 5.2  # Roraima rectangle
ANO = MES[:4]                        # folder year follows the month

arqs = glob.glob(f"{PASTA}/{ANO}/**/*.csv", recursive=True)
print("files found:", len(arqs))

df = pd.concat((pd.read_csv(a) for a in arqs), ignore_index=True)

# Filters: month, box, nominal/high confidence, vegetation fire
df = df[df["acq_date"].astype(str).str.startswith(MES)]
df = df[df["latitude"].between(S, N) & df["longitude"].between(W, E)]
df = df[df["confidence"].astype(str).str.lower().isin(["n", "h"])]
if "type" in df.columns:
    df = df[df["type"] == 0]

# Fire-day = distinct 0.01° cell with >= 1 detection on a given day
df["cy"] = (df["latitude"] / 0.01).round().astype(int)
df["cx"] = (df["longitude"] / 0.01).round().astype(int)
fire_days = df.drop_duplicates(["acq_date", "cy", "cx"]).shape[0]

print(f"VIIRS NOAA-20 detections:  {len(df):,}")
print(f"VIIRS fire-days, Roraima:  {fire_days:,}")

# Optional: fire-days per day
por_dia = df.drop_duplicates(["acq_date", "cy", "cx"]).groupby("acq_date").size()
print(por_dia)

**Cell: fire-days of one week in Roraima (ISO week 15 of 2024, 8 to 14 April).**
It uses local time (UTC-3). Result: **941 fire-days** with the ISO week, and **867** with the project week
(blocks of 7 days from 2000-01-01, which start on Saturday).
The site showed about 1,600, so its definition is different (confidence, sensors, week or cell size).
That is the point of the checks in the next cells.

In [ ]:
import glob, os
import pandas as pd

PASTA = os.path.expanduser("~/firms_brazil/viirs-jpss1")
ANO = 2024
SEMANA_ISO = 15
W, S, E, N = -64.8, -1.5, -59.0, 5.2  # Roraima (box)

arqs = glob.glob(f"{PASTA}/{ANO}/**/*.csv", recursive=True)
df = pd.concat((pd.read_csv(a) for a in arqs), ignore_index=True)

# Project filters
df = df[df["confidence"].astype(str).str.lower().isin(["n", "h"])]
if "type" in df.columns:
    df = df[df["type"] == 0]

# Roraima
df = df[df["latitude"].between(S, N) & df["longitude"].between(W, E)]

# Local date (UTC-3)
hhmm = df["acq_time"].astype(int).astype(str).str.zfill(4)
t = pd.to_datetime(df["acq_date"].astype(str) + " " + hhmm, format="%Y-%m-%d %H%M")
df["dia"] = (t - pd.Timedelta(hours=3)).dt.normalize()

# Fire-days: distinct 0.01° cell per day
df["cy"] = (df["latitude"] / 0.01).round().astype(int)
df["cx"] = (df["longitude"] / 0.01).round().astype(int)
cd = df.drop_duplicates(["dia", "cy", "cx"]).copy()

# ISO week
cd["sem_iso"] = cd["dia"].dt.isocalendar().week
sel = cd[cd["sem_iso"] == SEMANA_ISO]
print(f"Roraima, semana ISO {SEMANA_ISO}: {len(sel)} fire-days")
print(sel.groupby("dia").size())

# Project week (blocks of 7 days since 2000-01-01)
cd["sem_proj"] = (cd["dia"] - pd.Timestamp("2000-01-01")).dt.days // 7
sp = sel["dia"].map(lambda d: (d - pd.Timestamp("2000-01-01")).days // 7).mode()[0]
print(f"Semana do projeto {sp}: {(cd['sem_proj'] == sp).sum()} fire-days")

## 10. Export the filtered detections (one row per detection)

`pos_*.parquet` are already aggregated by cell and week (`cy, cx, week, n_det, frp_sum, frp_max`),
so they cannot be used when someone needs the individual detections.

**Cell: check what the `pos_*` files are.** Result: the columns are `cy, cx, week, n_det, frp_sum, frp_max`,
so they are aggregated by cell and week (not individual detections).

In [ ]:
import os, pandas as pd
for n in ["pos_modis_terra", "pos_viirs-jpss1"]:
    d = pd.read_parquet(os.path.expanduser(f"~/firms_output/{n}.parquet"))
    print(n, d.shape)
    print(d.head(3))

**Cell: check the filter values in the raw MODIS file of 2024.**
`satellite` has `Terra` and `Aqua`; `confidence` is numeric (0 to 100); `type` has 0, 2 and 3.
So the filters `satellite == Terra`, `confidence >= 30` and `type == 0` work as planned.

In [ ]:
import glob, os, pandas as pd
BASE = os.path.expanduser("~/firms_brazil")
a = glob.glob(f"{BASE}/modis/2024/**/*.csv", recursive=True)
d = pd.concat((pd.read_csv(x) for x in a), ignore_index=True)
print(d["satellite"].unique())
print(d["confidence"].describe())
print(d["type"].value_counts() if "type" in d.columns else "sem coluna type")

**Cell: compare the aggregated total with the filtered detections.**
It failed with `FileNotFoundError` because it ran before the export cell (the file did not exist yet).
Run it again after the export finishes: the two numbers must be equal.

In [ ]:
pos = pd.read_parquet(os.path.expanduser("~/firms_output/pos_modis_terra.parquet"))
f = pd.read_parquet(os.path.expanduser("~/firms_output/focos_filtrados/focos_modis_terra.parquet"))
print("n_det agregado:", pos["n_det"].sum())
print("focos filtrados:", len(f))

**Cell: install `psutil`** (to check the free memory). Only 1.3 GB of 8.2 GB were free,
so the export runs one year at a time.

In [ ]:
!pip install psutil

**Cell: export the filtered detections, one Parquet file per year.**
Same filters as the aggregation step, and a new column `data_hora_brt` (Brasilia time, UTC-3).
Each year is processed alone and then freed from memory.
Results (rows before -> after the filter):
- MODIS Terra 2000-2025: **3,521,179** detections kept (Aqua and type 2/3 removed).
- VIIRS NOAA-20 2018-2025: **9,711,325** detections kept.

Check: the per-year counts match the aggregation step (for example MODIS 2024: 193,310 and VIIRS 2024: 1,801,886).
Files: `~/firms_output/focos_filtrados/focos_modis_terra_<year>.parquet` and `focos_viirs_noaa20_<year>.parquet`.

In [ ]:
import glob, os, gc
import pandas as pd

BASE = os.path.expanduser("~/firms_brazil")
SAIDA = os.path.expanduser("~/firms_output/focos_filtrados")
os.makedirs(SAIDA, exist_ok=True)

def f_modis(d):
    # MODIS: Terra only, confidence >= 30, vegetation fire
    d = d[d["satellite"].astype(str).str.lower() == "terra"]
    d = d[pd.to_numeric(d["confidence"], errors="coerce") >= 30]
    if "type" in d.columns:
        d = d[d["type"] == 0]
    return d

def f_viirs(d):
    # VIIRS: nominal or high confidence, vegetation fire
    d = d[d["confidence"].astype(str).str.lower().isin(["n", "h"])]
    if "type" in d.columns:
        d = d[d["type"] == 0]
    return d

def processa(pasta_sensor, nome, anos, filtro):
    total = 0
    for ano in anos:
        arqs = glob.glob(f"{BASE}/{pasta_sensor}/{ano}/**/*.csv", recursive=True)
        if not arqs:
            print(nome, ano, "sem arquivos")
            continue
        d = pd.concat((pd.read_csv(a) for a in arqs), ignore_index=True)
        n0 = len(d)
        d = filtro(d)
        # local time (UTC-3)
        hhmm = d["acq_time"].astype(int).astype(str).str.zfill(4)
        t = pd.to_datetime(d["acq_date"].astype(str) + " " + hhmm, format="%Y-%m-%d %H%M")
        d["data_hora_brt"] = t - pd.Timedelta(hours=3)
        d.to_parquet(f"{SAIDA}/{nome}_{ano}.parquet", index=False)
        total += len(d)
        print(nome, ano, f"{n0:,} -> {len(d):,}")
        del d, t, hhmm
        gc.collect()          # free the memory before the next year
    print(nome, "TOTAL:", f"{total:,}")
    return total

n_modis = processa("modis", "focos_modis_terra", range(2000, 2026), f_modis)
n_viirs = processa("viirs-jpss1", "focos_viirs_noaa20", range(2018, 2026), f_viirs)

**Cell: zip the filtered detections** to send to a teammate. Result: a 222 MB file,
`~/firms_output/focos_filtrados.zip`.

In [ ]:
import shutil, os
pasta = os.path.expanduser("~/firms_output/focos_filtrados")
shutil.make_archive(os.path.expanduser("~/firms_output/focos_filtrados"), "zip", pasta)
print(os.path.getsize(pasta + ".zip") / 1e6, "MB")